In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voebem/bronze/arquivos/referencias"
TABELA = "voebem.bronze.aerodromos"

# Caractere que NÃO existe no arquivo -> desliga o quoting do leitor de CSV
SEM_ASPAS = chr(0)

In [0]:
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("encoding", "ISO-8859-1")   
    .option("quote", SEM_ASPAS)         
    .load(f"{REF}/AerodromosPublicos.csv")
)

aerodromos.columns

## Seleção e renomeação de colunas

Utilização do método `select()` com expressões `F.col().alias()` para realizar a filtragem e a padronização dos nomes das colunas em um único passo:
* **Filtragem de Colunas**: Mantém apenas os campos relevantes para a camada Bronze, descartando colunas desnecessárias do CSV original.
* **Padronização (`snake_case`)**: Renomeia colunas com acentos, espaços e caracteres especiais para nomes padronizados em minúsculas.
* **Auditoria**: Adiciona o campo `_ingerido_em` com o timestamp atual para rastreabilidade do momento da carga.

In [0]:
aerodromos = aerodromos.select(
    F.col("Código OACI").alias("icao"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("Município").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("Município Servido").alias("municipio_servido"),
    F.col("UF Servido").alias("uf_servido"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("Situação").alias("situacao"),
).withColumn("_ingerido_em", F.current_timestamp())

display(aerodromos)

## Persistência no Delta Lake (Carga Full Refresh)
Gravação do DataFrame no formato **Delta Lake**, aplicando a estratégia de recarga completa (*overwrite*):
* **`mode("overwrite")`** : Substitui o conteúdo existente pelos novos dados processados.
* **`option("overwriteSchema", "true")`** : Permite a atualização automática do esquema da tabela caso novas colunas sejam adicionadas futuramente.

In [0]:
(
    aerodromos.write.format("delta")
    .mode("overwrite")  
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

## Gestão de metadados

Documentação e verificação da qualidade da carga executada na tabela Bronze:
* **Documentação do Catálogo**: Adição do comentário descritivo no Unity Catalog/Hive Metastore informando a regra de negócio, origem e histórico dos dados.
* **Validação de Ingestão**: Consulta de agregação para conferir a contagem total de registros e o horário da última carga executada.

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS
          'Bronze - Referência de Aeródromos da ANAC.
          Dados de cadastro e localização dos aeródromos públicos brasileiros.
          Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/referencias'
          """)

In [0]:
display(
    spark.sql(f"""
              SELECT count(*) AS total_aerodromos, max(_ingerido_em) AS ingerido_em
              FROM {TABELA}
              """)
)

## Função reutilizável de ingestão 

Criação da função utilitária `ler_empresas()` para automatizar a leitura, padronização e estruturação das bases de empresas aéreas (nacionais e estrangeiras):

In [0]:
def ler_empresas(arquivo: str):
    return (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{REF}/{arquivo}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("Ativa").alias("situacao"),
        )
        .withColumn("_arquivo_origem", F.lit(arquivo))
        .withColumn("_ingerido_em", F.current_timestamp())
    )

for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv",    "voebem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_estrangeiras"),
]:
    ler_empresas(arquivo).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(tabela)

In [0]:
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voebem.bronze.empresas_estrangeiras
"""))

## Visualização e validação das empresas nacionais e estrangeiras
- **Propósito**: Filtra e exibe os dados cadastrais das principais companhias aéreas brasileiras e internacionais para confirmar a integridade e a padronização das colunas.

In [0]:
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
"""))

display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
"""))

## Códigos de Operação do VRA
  - **`codigo_di`**: Decodifica o tipo de etapa do voo (ex.: Etapa Regular, Extra, Fretamento).
  - **`codigo_tipo_linha`**: Classifica o serviço prestado na linha aérea (ex.: Doméstica Mista, Internacional Cargueira).

In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")
codigos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.codigos_operacao")

display(
    spark.table("voebem.bronze.codigos_operacao")
)

In [0]:
display(spark.sql("SHOW TABLES IN voebem.bronze"))